# Notebook 4 — The Economics of Cheap Prediction
**AI Models and Agents for Business · Week 2 · Session 4**

The *Prediction Machines* argument in one line: **AI is a drop in the cost of prediction**, and when the price of an input falls far enough, you don't just buy more of it — you start using it in places it never made sense before.

This notebook turns that claim into arithmetic. It's the most business-flavored notebook in the course: no model training, no API calls, just the decision economics that determine whether an AI deployment makes money.

**No API key needed.**

| Part | The question |
|---|---|
| A | A prediction is not a decision. What has to be added? |
| B | What happens to the *optimal threshold* when the payoff matrix changes? |
| C | As prediction gets cheaper, which decisions become worth automating? |
| D | Where does human judgment stay expensive — and therefore valuable? |

**Setting:** you run fraud operations at a payments company. Every transaction can be auto-approved, or held for manual review.

## Cell 1 — Setup and a stream of transactions

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.RandomState(21)
N = 20_000
FRAUD_RATE = 0.015          # 1.5% of transactions are fraudulent

is_fraud = rng.random(N) < FRAUD_RATE
amount = np.round(rng.lognormal(3.9, 1.05, N), 2).clip(5, 5000)

# The model sees a risk signal. It's good but not perfect:
# fraudulent transactions score higher on average, with real overlap.
signal = rng.normal(np.where(is_fraud, 2.1, 0.0), 1.0)

# Turn the signal into a *calibrated* probability: among transactions with fraud_prob = 0.20,
# about 20% really are fraud. This uses Bayes' rule with the 1.5% base rate. (Squashing the
# raw signal through 1 / (1 + e^-signal) would rank transactions the same way, but the
# numbers would not be probabilities: legitimate transactions would average about 0.5.)
log_odds = np.log(FRAUD_RATE / (1 - FRAUD_RATE)) + 2.1 * signal - 2.1**2 / 2
fraud_prob = 1 / (1 + np.exp(-log_odds))

txns = pd.DataFrame({"amount": amount, "fraud_prob": fraud_prob.round(4), "is_fraud": is_fraud})

print(f"Transactions      : {N:,}")
print(f"Actually fraud    : {is_fraud.sum():,} ({is_fraud.mean():.2%})")
print(f"Total value       : ${amount.sum():,.0f}")
print(f"Fraud exposure    : ${amount[is_fraud].sum():,.0f}")
print()
print("Model separation check (mean predicted probability):")
print(f"  legitimate transactions : {fraud_prob[~is_fraud].mean():.3f}")
print(f"  fraudulent transactions : {fraud_prob[is_fraud].mean():.3f}")
print()
print("Calibration check (do the probabilities mean what they say?):")
print(f"  average predicted probability : {fraud_prob.mean():.2%}")
print(f"  actual fraud rate             : {is_fraud.mean():.2%}")
txns.head()

---
## Part A: A prediction is not a decision

The model gives you a **probability**. That's all it gives you. To turn it into an action you must supply four numbers the model cannot know:

| Outcome | What it costs you |
|---|---|
| Correctly approve a legitimate transaction | you earn the fee |
| Hold a legitimate transaction for review (**false positive**) | review cost + annoyed customer |
| Approve a fraudulent transaction (**false negative**) | you eat the fraud loss |
| Correctly hold a fraudulent transaction | review cost, but loss avoided |

Those numbers are **judgment**. They come from your business, your margins, your tolerance for angering customers. *Prediction Machines*' central structural claim is that prediction and judgment are **complements** — cheap prediction makes judgment more valuable, not less, because judgment is now the binding constraint.

In [ ]:
# ✏️ CHANGE ME — this payoff structure IS the judgment layer
FEE_RATE           = 0.025   # you earn 2.5% on an approved legitimate transaction
REVIEW_COST        = 4.00    # analyst time to manually review one held transaction
FALSE_ALARM_COST   = 6.00    # goodwill/churn cost of wrongly holding a good customer
FRAUD_RECOVERY     = 0.15    # you recover 15% of a fraud loss after the fact

def evaluate(threshold, data):
    """Total profit if we hold every transaction scoring above `threshold`.

    The four judgment values are read fresh on each call, so editing them above
    and re-running immediately changes the answer.
    """
    held = data.fraud_prob >= threshold
    approved = ~held

    revenue        = (data.amount[approved & ~data.is_fraud] * FEE_RATE).sum()
    fraud_losses   = (data.amount[approved & data.is_fraud] * (1 - FRAUD_RECOVERY)).sum()
    review_costs   = held.sum() * REVIEW_COST
    false_alarms   = (held & ~data.is_fraud).sum() * FALSE_ALARM_COST

    return {
        "threshold": threshold,
        "held": int(held.sum()),
        "caught_fraud": int((held & data.is_fraud).sum()),
        "missed_fraud": int((approved & data.is_fraud).sum()),
        "false_alarms": int((held & ~data.is_fraud).sum()),
        "revenue": revenue,
        "fraud_losses": fraud_losses,
        "review_costs": review_costs,
        "false_alarm_costs": false_alarms,
        "profit": revenue - fraud_losses - review_costs - false_alarms,
    }

for t in [0.0, 0.5, 1.01]:
    r = evaluate(t, txns)
    name = {0.0: "Review EVERYTHING", 0.5: "Review if p >= 0.50", 1.01: "Review NOTHING"}[t]
    print(f"{name:<22} held {r['held']:>6,}  caught {r['caught_fraud']:>4}  "
          f"missed {r['missed_fraud']:>4}  profit ${r['profit']:>12,.0f}")

### 💬 Discussion — Part A
- "Review everything" catches all fraud. Why isn't it the right answer?
- "Review nothing" has zero review cost. Why isn't *that* the right answer?
- The model was identical in all three rows. What changed? *(Only the decision rule — i.e. the judgment applied to the prediction.)*
- Which of the four cost numbers above would be hardest to get agreement on inside a real company, and who would you need in the room?

---
## Part B: Finding the optimal threshold — and watching judgment move it

Now sweep every threshold and find the profit-maximizing one. Then change the *judgment* (not the model) and watch the answer move.

In [ ]:
thresholds = np.round(np.arange(0.0, 1.001, 0.01), 3)
results = pd.DataFrame([evaluate(t, txns) for t in thresholds])
best = results.loc[results.profit.idxmax()]

fig, ax = plt.subplots(1, 2, figsize=(14, 4.6))
ax[0].plot(results.threshold, results.profit)
ax[0].axvline(best.threshold, linestyle="--", alpha=0.7,
              label=f"Optimum p >= {best.threshold:.2f}")
ax[0].set_xlabel("Hold-for-review threshold"); ax[0].set_ylabel("Total profit ($)")
ax[0].set_title("Profit is a curve, not a cliff"); ax[0].legend(); ax[0].grid(alpha=0.3)

ax[1].plot(results.threshold, results.fraud_losses, label="Fraud losses (missed)")
ax[1].plot(results.threshold, results.review_costs + results.false_alarm_costs,
           label="Review + false-alarm costs")
ax[1].axvline(best.threshold, linestyle="--", alpha=0.7)
ax[1].set_xlabel("Hold-for-review threshold"); ax[1].set_ylabel("Cost ($)")
ax[1].set_title("The two costs you're trading off"); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f"Optimal threshold : p >= {best.threshold:.2f}")
print(f"  transactions held : {best.held:,.0f} of {N:,} ({best.held/N:.1%})")
print(f"  fraud caught      : {best.caught_fraud:,.0f}")
print(f"  fraud missed      : {best.missed_fraud:,.0f}")
print(f"  profit            : ${best.profit:,.0f}")

## Cell 4 — Same model, different judgment

Below we hold the model completely fixed and change only the business's priorities. Watch the optimal threshold move.

In [ ]:
scenarios = {
    "Baseline":                      dict(),
    "Regulator watching (fraud 3x costlier)":  dict(FRAUD_RECOVERY=-2.0),
    "Cheap offshore review team":     dict(REVIEW_COST=1.0, FALSE_ALARM_COST=6.0),
    "Premium customers (alarms hurt)": dict(FALSE_ALARM_COST=40.0),
    "Thin margins (fee halved)":      dict(FEE_RATE=0.0125),
}

print(f"{'Scenario':<42}{'Best threshold':<17}{'% held':<10}{'Profit'}")
print("-" * 82)

_defaults = dict(FEE_RATE=FEE_RATE, REVIEW_COST=REVIEW_COST,
                 FALSE_ALARM_COST=FALSE_ALARM_COST, FRAUD_RECOVERY=FRAUD_RECOVERY)

for label, overrides in scenarios.items():
    cfg = {**_defaults, **overrides}
    FEE_RATE, REVIEW_COST, FALSE_ALARM_COST, FRAUD_RECOVERY = (
        cfg["FEE_RATE"], cfg["REVIEW_COST"], cfg["FALSE_ALARM_COST"], cfg["FRAUD_RECOVERY"])
    res = pd.DataFrame([evaluate(t, txns) for t in thresholds])
    b = res.loc[res.profit.idxmax()]
    print(f"{label:<42}p >= {b.threshold:<12.2f}{b.held/N:<10.1%}${b.profit:>12,.0f}")

# restore baseline
FEE_RATE, REVIEW_COST, FALSE_ALARM_COST, FRAUD_RECOVERY = (
    _defaults["FEE_RATE"], _defaults["REVIEW_COST"],
    _defaults["FALSE_ALARM_COST"], _defaults["FRAUD_RECOVERY"])
print()
print("The model never changed. Only the judgment did.")

### 💬 Discussion — Part B
- The prediction model is byte-for-byte identical across every row. Yet the right operating policy differs substantially. What does that tell you about where the value sits in an AI deployment?
- Your data science team says "we improved AUC from 0.91 to 0.93." Your ops team says "we renegotiated review costs from $4 to $1." Using the table above, which had more impact on profit?
- The best baseline threshold is far below 50%. Why would you hold a transaction that the model says is probably *legitimate*? *(Compare what a missed fraud costs with what a review costs.)*
- That reasoning only works because these probabilities are **calibrated**: a 0.14 really means about a 14% chance of fraud. Many models output a *score* that ranks cases well but isn't a probability. What would go wrong if you plugged a raw score into this cost math?
- Data scientists often tune a threshold by counting errors alone (frauds caught, false alarms raised). This is the same exercise **with the dollar signs**. Which version would you put in front of a CFO?

---
## Part C: What happens as prediction gets cheap

Here is the *Prediction Machines* thesis stated as an experiment.

Before machine learning, "predicting" whether a transaction was fraudulent meant **a human analyst looking at it** — slow and expensive. Now a model scores it for a fraction of a cent. So: at what price per prediction does it become worth scoring **every** transaction rather than only the large ones?

In [ ]:
# Strategy A: only score transactions above a value cutoff (prediction is expensive,
#             so you ration it). Unscored transactions are auto-approved.
# Strategy B: score everything (prediction is nearly free).

def profit_with_selective_prediction(price_per_prediction, value_cutoff, data,
                                     threshold=None):
    threshold = best.threshold if threshold is None else threshold
    scored = data.amount >= value_cutoff

    # Scored transactions: use the model + optimal threshold
    s = data[scored]
    held = s.fraud_prob >= threshold
    rev_s   = (s.amount[~held & ~s.is_fraud] * FEE_RATE).sum()
    loss_s  = (s.amount[~held & s.is_fraud] * (1 - FRAUD_RECOVERY)).sum()
    rc_s    = held.sum() * REVIEW_COST
    fa_s    = (held & ~s.is_fraud).sum() * FALSE_ALARM_COST

    # Unscored transactions: auto-approved blind, so all their fraud lands on you
    u = data[~scored]
    rev_u  = (u.amount[~u.is_fraud] * FEE_RATE).sum()
    loss_u = (u.amount[u.is_fraud] * (1 - FRAUD_RECOVERY)).sum()

    prediction_spend = scored.sum() * price_per_prediction
    return rev_s + rev_u - loss_s - loss_u - rc_s - fa_s - prediction_spend

price_points = [8.00, 2.00, 0.50, 0.10, 0.01, 0.0002]
price_labels = ["$8.00 (human analyst)", "$2.00", "$0.50", "$0.10",
                "$0.01", "$0.0002 (LLM API)"]
cutoffs = [0, 25, 50, 100, 250, 500, 1000]

print("Profit by prediction price and how much you ration predictions:")
print(f"{'Price/prediction':<24}" + "".join(f"{'>=$'+str(c):>11}" for c in cutoffs) + "   Best")
print("-" * 106)
for price, lab in zip(price_points, price_labels):
    row = [profit_with_selective_prediction(price, c, txns) for c in cutoffs]
    bestc = cutoffs[int(np.argmax(row))]
    cells = "".join(f"{v/1000:>10,.1f}k" for v in row)
    print(f"{lab:<24}{cells}   score all >= ${bestc}")

### 💬 Discussion — Part C
- Read down the "Best" column. As the price per prediction falls, what happens to the value cutoff you should use?
- At analyst prices ($8), prediction is a scarce resource you spend only on transactions large enough to justify it — the cutoff sits at $1,000. By API prices it has collapsed to $25. **This is the whole thesis:** cheap prediction doesn't just reduce the cost of predictions you were already making, it makes prediction worth doing in places it previously wasn't.
- Look closely at the cheapest two rows: scoring *everything* (`>=$0`) is still very slightly worse than `>=$25`. Why? Because the prediction is nearly free but the **review** it triggers still costs $4. Once prediction cost goes to roughly zero, the binding constraint stops being prediction and becomes the cost of the *action* the prediction sets off. That's an easy thing to miss when a vendor pitches you on cost-per-call.
- Name a business decision at a company you know that is *currently* made without any prediction at all, purely because predicting was too expensive. What would change if prediction were free?
- What does this framework say about *where the profit went*? If prediction is now nearly free and available to all your competitors, is it a source of competitive advantage — or is it becoming table stakes, with advantage moving to judgment and proprietary data?

## Cell 6 — ✏️ CHANGE ME: your own scenario

Set the four judgment numbers for a decision you actually care about, and find its optimal policy.

In [ ]:
# ✏️ CHANGE ME — model your own decision
FEE_RATE         = 0.025
REVIEW_COST      = 4.00
FALSE_ALARM_COST = 6.00
FRAUD_RECOVERY   = 0.15

my_results = pd.DataFrame([evaluate(t, txns) for t in thresholds])
my_best = my_results.loc[my_results.profit.idxmax()]

do_nothing = evaluate(1.01, txns)["profit"]
uplift = my_best.profit - do_nothing

print(f"Optimal policy : hold for review when p >= {my_best.threshold:.2f}")
print(f"Profit         : ${my_best.profit:,.0f}")
print(f"vs. no model   : ${do_nothing:,.0f}")
print(f"Value of the model itself: ${uplift:,.0f} over {N:,} transactions")
print(f"                        = ${uplift/N:.4f} per transaction")
print()
print(f"So you'd pay up to ${uplift/N:.4f} per prediction before the model stops paying for itself.")
print("Compare that to what an LLM API call actually costs.")

### 💬 Discussion — Part D: where judgment stays expensive
Apply the framework to the cases you study next, in Sessions 5 and 6. Read them with these questions in mind:

- **Zillow (Session 5):** prediction got cheap, so Zillow predicted home values at enormous scale. Which part of the framework did they get wrong — the prediction, or the judgment about what to do when the prediction was uncertain?
- **Klarna (Session 6):** cheap prediction made automated support viable. What judgment call did the unit economics miss?
- **COMPAS (Session 5):** the prediction was cheap and reasonably accurate. Was the controversy about the prediction, or about the payoff matrix nobody agreed on?
- **Amazon (Session 5):** the model predicted "resembles our past hires" very well. Whose judgment defined the objective, and what did that judgment encode?

The claim to test as you read: in every case the prediction worked roughly as designed, and the failure lived in the judgment layer. That's the strongest practical argument for why business students — not just engineers — need to be in these rooms.

## ✅ Notebook 4 complete — Reflection questions

These prep you for **Quiz 1** (Session 5).

1. State the *Prediction Machines* thesis in one sentence, in your own words.
2. What is "judgment" in this framework, and why can't the model supply it?
3. Part B changed the optimal threshold without touching the model. Explain to a non-technical executive why that happened.
4. Why does cheap prediction *increase* the value of judgment and proprietary data rather than reducing it?
5. Your model generates $0.03 of value per prediction and the API charges $0.0002 per call. What's the constraint on scaling this to every decision in the company — and is it a technical constraint or an organizational one?

---
*Next session: Quiz 1, then Prediction as Process Automation: the Zillow, Amazon and COMPAS cases.*